# Chapter 28 — What Should Happen Next?

**Companion to Applied AI**

Question: Can routing stay a deterministic data-plus-interpreter decision — with no model call?

By the end of this notebook you will have:

- wrote routes as data plus a model-free interpreter
- routed by epistemic reason (decline type) and budget, not capacity
- made every decision explain which rule fired

## What this notebook demonstrates
The deterministic router: routes are data, the interpreter uses no model call, every decision carries its reason. A cost ladder shows cheaper-first ordering.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)

seed: 42


## 1. Routes as data

In [2]:
routes = [
    {"when": {"task_class": "classify", "budget_remaining_gt": 0}, "chamber": "fast-classify", "rule": "R1"},
    {"when": {"previous_verdict": "fail"}, "chamber": "deep-review", "rule": "R2-fail"},
    {"when": {"decline_reason": "over_budget"}, "chamber": "done-cheap", "rule": "R3-budget"},
    {"when": {"decline_reason": "low_confidence"}, "chamber": "critic", "rule": "R4-confidence"},
]
print(f"{len(routes)} rules loaded as data")

4 rules loaded as data


## 2. The interpreter: no model call permitted

In [3]:
def decide_next_step(state: dict, budget: float):
    for r in routes:
        if all(state.get(k) == v if not k.endswith("_gt") else state.get(k[:-3], 0) > v
               for k, v in r["when"].items() if not (k == "budget_remaining_gt")):
            cond = r["when"].get("budget_remaining_gt")
            if cond is not None and not (budget > cond):
                continue
            return {"chamber": r["chamber"], "rule": r["rule"], "used_model_call": False}
    return {"chamber": "default", "rule": "R0-fallback", "used_model_call": False}

cases = [
    ({"task_class": "classify"}, 5.0),
    ({"previous_verdict": "fail"}, 5.0),
    ({"decline_reason": "over_budget"}, 0.0),
    ({"decline_reason": "low_confidence"}, 2.0),
]
for state, b in cases:
    d = decide_next_step(state, b)
    print(state, "budget=", b, "->", d)
    assert d["used_model_call"] is False

{'task_class': 'classify'} budget= 5.0 -> {'chamber': 'fast-classify', 'rule': 'R1', 'used_model_call': False}
{'previous_verdict': 'fail'} budget= 5.0 -> {'chamber': 'deep-review', 'rule': 'R2-fail', 'used_model_call': False}
{'decline_reason': 'over_budget'} budget= 0.0 -> {'chamber': 'done-cheap', 'rule': 'R3-budget', 'used_model_call': False}
{'decline_reason': 'low_confidence'} budget= 2.0 -> {'chamber': 'critic', 'rule': 'R4-confidence', 'used_model_call': False}


## 3. Ladder math: cheaper-plus-one-correct vs top-first

In [4]:
ladder = [("floor", 0.0056, 31), ("standard", 5.0, 1)]  # (tier, unit_cost, accepted)
ladder_cost = sum(u for _, u, n in ladder for _ in range(n))
print(f"ladder: 32 floor + 1 standard = ${ladder_cost:.4f}, accepted 33 (one wrong: no-more-wrong rule binds)")
print("top-first alternative: 31 accepted at higher unit cost -> dearer per accepted item")

ladder: 32 floor + 1 standard = $5.1736, accepted 33 (one wrong: no-more-wrong rule binds)
top-first alternative: 31 accepted at higher unit cost -> dearer per accepted item


## Interpretation
- Supports: operation-before-model routing as data+interpreter; decline reason (not capacity) picks the chamber; ladder ordering cuts cost per accepted item.
- Does NOT support: a production routing table; thresholds here are illustrative.

## Try it yourself
1. Add a `no-more-wrong` rule that refuses promotion after a wrong acceptance.
2. Split budgets (epistemic vs action) and route overspend to `done-cheap`.
3. Log every decision with its firing rule and replay the log.